<a href="https://colab.research.google.com/github/codingwithp/Innomatics-Research-Labs/blob/main/NLP_Task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")
print("Path:", path)

100%|██████████| 1.99M/1.99M [00:00<00:00, 84.0MB/s]

Extracting files...
Path: /root/.cache/kagglehub/datasets/jp797498e/twitter-entity-sentiment-analysis/versions/2


In [3]:
import pandas as pd

file_path = path + "/twitter_training.csv"

df = pd.read_csv(file_path)
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [2]:
import os

print(os.listdir(path))

['twitter_training.csv', 'twitter_validation.csv']


In [4]:
df.columns = ["id", "entity", "sentiment", "text"]
df.head()

,id,entity,sentiment,text
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [5]:
df = df[["text", "sentiment"]]
df.head()


,text,sentiment
0,I am coming to the borders and I will kill you...,Positive
1,im getting on borderlands and i will kill you ...,Positive
2,im coming on borderlands and i will murder you...,Positive
3,im getting on borderlands 2 and i will murder ...,Positive
4,im getting into borderlands and i can murder y...,Positive


In [6]:
df = df.dropna()

In [7]:
print(df['sentiment'].value_counts())

sentiment
Negative      22358
Positive      20654
Neutral       18108
Irrelevant    12875
Name: count, dtype: int64


In [8]:
df = df[df['sentiment'].isin(['Positive', 'Negative', 'Neutral'])]

In [9]:
df['sentiment'] = df['sentiment'].str.lower()

In [10]:
print(df.shape)
print(df['sentiment'].value_counts())
df.head()

(61120, 2)
sentiment
negative    22358
positive    20654
neutral     18108
Name: count, dtype: int64


,text,sentiment
0,I am coming to the borders and I will kill you...,positive
1,im getting on borderlands and i will kill you ...,positive
2,im coming on borderlands and i will murder you...,positive
3,im getting on borderlands 2 and i will murder ...,positive
4,im getting into borderlands and i can murder y...,positive


In [11]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)

    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]

    return " ".join(words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [12]:
df['clean_text'] = df['text'].apply(preprocess)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['clean_text'])  # features
y = df['sentiment']                        # labels

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [16]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

In [17]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

In [18]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_test, y_pred):
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Logistic Regression")
evaluate(y_test, y_pred_lr)

print("\nNaive Bayes")
evaluate(y_test, y_pred_nb)

print("\nDecision Tree")
evaluate(y_test, y_pred_dt)

Logistic Regression
Accuracy: 0.7694698952879581
Precision: 0.7702916578345282
Recall: 0.7694698952879581
F1 Score: 0.7680971896209399

Naive Bayes
Accuracy: 0.7248854712041884
Precision: 0.7295785925320133
Recall: 0.7248854712041884
F1 Score: 0.7205337968259253

Decision Tree
Accuracy: 0.8294339005235603
Precision: 0.8307845862121592
Recall: 0.8294339005235603
F1 Score: 0.8295755570688712


In [19]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ]
})

results.sort_values(by="Accuracy", ascending=False)

,Model,Accuracy
2,Decision Tree,0.829434
0,Logistic Regression,0.769470
1,Naive Bayes,0.724885


In [20]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)

lr_bow = LogisticRegression(max_iter=200)
lr_bow.fit(X_train_bow, y_train)

y_pred_bow = lr_bow.predict(X_test_bow)

bow_acc = accuracy_score(y_test, y_pred_bow)
print("BoW Accuracy:", bow_acc)
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

lr_tfidf = LogisticRegression(max_iter=200)
lr_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = lr_tfidf.predict(X_test_tfidf)

tfidf_acc = accuracy_score(y_test, y_pred_tfidf)
print("TF-IDF Accuracy:", tfidf_acc)

BoW Accuracy: 0.7864856020942408
TF-IDF Accuracy: 0.7694698952879581


In [23]:
print("BoW Accuracy:", bow_acc)
print("TF-IDF Accuracy:", tfidf_acc)

BoW Accuracy: 0.7864856020942408
TF-IDF Accuracy: 0.7694698952879581


Model Comparison

The Decision Tree model performed the best with an accuracy of 0.829.

Logistic Regression achieved an accuracy of 0.769, showing moderate performance.

Naive Bayes had the lowest accuracy of 0.724, likely due to its assumption of feature independence.

 TF-IDF vs Bag of Words

Bag of Words achieved an accuracy of 0.786, while TF-IDF achieved 0.769.

In this case, Bag of Words performed slightly better than TF-IDF, which indicates that simple word frequency worked better for this dataset.

Trade-offs
Decision Tree → Highest accuracy but may overfit
Logistic Regression → More stable and generalizable
Naive Bayes → Fast but less accurate